# 01 · 错误链与表格测试

目标：练习 `%w`、`errors.Is/As`、结构化错误与表格用例。对应文档：`01_language_basics/1.2-error-handling.md`、`1.5-testing.md`。

## 从 TypeScript 迁移

Go 的 `error` 是显式返回值，不会像 throw/rejected Promise 跳过调用链。`%w` 类似保留 cause；`errors.Is` 检查类别，`errors.As` 提取结构化类型。

In [ ]:
import (
    "errors"
    "fmt"
    "strings"
)

var ErrUserNotFound = errors.New("user not found")

type ValidationError struct {
    Field  string
    Reason string
}

func (e *ValidationError) Error() string {
    return e.Field + ": " + e.Reason
}

func findUser(id string) (string, error) {
    if id != "u-1" {
        return "", fmt.Errorf("lookup user %q: %w", id, ErrUserNotFound)
    }
    return "Ada", nil
}

func renameUser(id, name string) error {
    if strings.TrimSpace(name) == "" {
        return &ValidationError{Field: "name", Reason: "must not be empty"}
    }
    _, err := findUser(id)
    if err != nil {
        return fmt.Errorf("rename user: %w", err)
    }
    return nil
}

## 实验一：检查错误链

先预测：包装两层以后，`errors.Is(err, ErrUserNotFound)` 是否仍为 true？为什么不应该比较 `err.Error()` 字符串？

In [ ]:
%%
errMissing := renameUser("missing", "Grace")
fmt.Println("完整错误链:", errMissing)
fmt.Println("errors.Is:", errors.Is(errMissing, ErrUserNotFound))

errValidation := renameUser("u-1", "  ")
var validation *ValidationError
fmt.Println("errors.As:", errors.As(errValidation, &validation))
fmt.Printf("结构化字段: field=%s reason=%s\n", validation.Field, validation.Reason)

`%w` 保留因果链，因此调用方可以判断稳定类别，又能看到 `rename user`、`lookup user` 等定位上下文。HTTP 边界应映射类别，不应把完整内部链直接返回客户端。

## 表格驱动的边界用例

Go 测试通常把输入、期望分类放进同一张表，再用 `t.Run` 执行。这里先在 Notebook 中观察同样的数据结构。

In [ ]:
%%
cases := []struct {
    name      string
    id        string
    newName   string
    wantError error
}{
    {name: "success", id: "u-1", newName: "Grace"},
    {name: "missing", id: "u-2", newName: "Grace", wantError: ErrUserNotFound},
    {name: "invalid name", id: "u-1", newName: "  ", wantError: &ValidationError{}},
}

for _, testCase := range cases {
    err := renameUser(testCase.id, testCase.newName)
    passed := err == nil && testCase.wantError == nil
    if errors.Is(testCase.wantError, ErrUserNotFound) {
        passed = errors.Is(err, ErrUserNotFound)
    }
    if _, ok := testCase.wantError.(*ValidationError); ok {
        var target *ValidationError
        passed = errors.As(err, &target)
    }
    fmt.Printf("%-14s pass=%t err=%v\n", testCase.name, passed, err)
}

## 验证真实服务

Notebook 中的表格便于观察结构；真正的 `testing.T`、子测试和 HTTP 错误映射由下面的 API Service 测试验证。

In [ ]:
!*cd "$GO_LEARNING_API_SERVICE" && go test ./internal/httpapi -run 'TestHandlerRejectsInvalidCreateRequests|TestHandlerReturnsNotFoundForMissingResource' -v

## 动手挑战

1. 增加 `PermissionError`，用 `errors.As` 提取用户和动作。
2. 把 `%w` 改成 `%v`，观察 `errors.Is` 失效。
3. 为临时错误设计 `Temporary() bool` 行为接口。

## 复盘

- sentinel、自定义类型和行为接口分别适合什么？
- 为什么同一错误通常只在系统边界记录一次？
- panic 为什么不适合普通业务失败？